In [ ]:
import polars as pl
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, root_mean_squared_error

from catboost import CatBoostRegressor

import optuna

import nbformat

import warnings

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet')
df.shape

In [ ]:
df.head()

In [ ]:
def objective(trial):
    params = {
        "iterations": 10_000,
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.5, log=True),
        "depth": trial.suggest_int("depth", 2, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 20.0, log=True),
        "border_count": trial.suggest_int("border_count", 1, 255),
        "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        "rsm": trial.suggest_float("rsm", 0.1, 1.0), 
        
        "feature_border_type": trial.suggest_categorical("feature_border_type", 
            ["Median", "Uniform", "UniformAndQuantiles", "MaxLogSum", "MinEntropy"]),

        "max_ctr_complexity": trial.suggest_int("max_ctr_complexity", 1, 5),
        "one_hot_max_size": trial.suggest_int("one_hot_max_size", 0, 25),
        
        "model_shrink_rate": trial.suggest_float("model_shrink_rate", 0, 1.0),
        "model_shrink_mode": trial.suggest_categorical("model_shrink_mode", ["Constant", "Decreasing"]),

        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "od_type": "Iter",
        "od_wait": 50,
        "verbose": False,
        "random_state": 42
    }
    bootstrap_type = trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS", "No"])
    params["bootstrap_type"] = bootstrap_type
    
    if bootstrap_type == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10)
    elif bootstrap_type in ["Bernoulli", "MVS"]:
        params["subsample"] = trial.suggest_float("subsample", 0.1, 1)
    
    if params["grow_policy"] == "Lossguide":
        params["max_leaves"] = trial.suggest_int("max_leaves", 16, 512)

    reg = CatBoostRegressor(**params,
    cat_features=['metro', 'title', 'author'],
    #text_features=['description', 'address', 'metro'],
    )

    reg.fit(X_train, Y_train, eval_set=[(X_val, Y_val)], early_stopping_rounds=200)

    return root_mean_squared_error(Y_val, reg.predict(X_val))


In [ ]:
X = df[['area', 'metro_time', 'photo_count', 
'self_floor', 'max_floor', 'metro', 'title', 
'author', 
# 'description', 'address'
]]

X['metro'] = X['metro'].apply(lambda x: str(x))
X['title'] = X['title'].apply(lambda x: str(x))
X['author'] = X['author'].apply(lambda x: str(x))
# X['description'] = X['description'].apply(lambda x: str(x))
# X['address'] = X['address'].apply(lambda x: str(x))


Y = df['price_numeric']

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, shuffle=True)

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100)

print(f"Лучший RMSE: {study.best_value}")
print(f"Параметры: {study.best_params}")

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
X = df[['area', 'metro_time', 'photo_count', 
'self_floor', 'max_floor', 'metro', 'title', 
'author', 'description', 'address'
]]

X['metro'] = X['metro'].apply(lambda x: str(x))
X['title'] = X['title'].apply(lambda x: str(x))
X['author'] = X['author'].apply(lambda x: str(x))
X['description'] = X['description'].apply(lambda x: str(x))
X['address'] = X['address'].apply(lambda x: str(x))


Y = df['price_numeric']

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.1, shuffle=True)

In [ ]:
params_2 = {'learning_rate': 0.03885692284397047, 
'depth': 8, 
'l2_leaf_reg': 9.604351731786379e-05, 
'random_strength': 0.15706222739732337, 
'border_count': 251, 
'grow_policy': 'Lossguide', 
'min_data_in_leaf': 21, 
'rsm': 0.45981943578435236, 
'feature_border_type': 'UniformAndQuantiles', 
'max_ctr_complexity': 1, 
'one_hot_max_size': 2, 
'model_shrink_rate': 0.7450959480799968, 
'model_shrink_mode': 'Decreasing', 
'bootstrap_type': 'No', 
'max_leaves': 379, 

'iterations': 10_000,
"loss_function": "RMSE",
"eval_metric": "RMSE",
"od_type": "Iter",
"od_wait": 50,
"verbose": 100,
"random_state": 42}

In [ ]:
best_params = {
'learning_rate': 0.07240029457505619,
 'depth': 8,
 'l2_leaf_reg': 0.47154458576232194,
 'random_strength': 14.618825667749789,
 'border_count': 253,
 'grow_policy': 'SymmetricTree',
 'min_data_in_leaf': 60,
 'rsm': 0.9964534252040771,
 'feature_border_type': 'Median',
 'max_ctr_complexity': 4,
 'one_hot_max_size': 22,
 'model_shrink_rate': 0.38764528904394,
 'model_shrink_mode': 'Decreasing',
 'bootstrap_type': 'No', 

'iterations': 5000,
"loss_function": "RMSE",
"eval_metric": "RMSE",
"od_type": "Iter",
"od_wait": 50,
"verbose": 100,
"random_state": 42}



In [ ]:
model = CatBoostRegressor(
**params_2,
cat_features=['metro', 'title', 'author'],
text_features=['description', 'address', 'metro'],
)

model.fit(X_train, Y_train, eval_set=[(X_val, Y_val)], early_stopping_rounds=100)

In [ ]:
model.save_model("catb_all_features_with_optim_60K.cbm")

In [ ]:
def eval_with_metrics(model, X, Y):
    
    preds = model.predict(X)

    print(f"R^2: {r2_score(Y, preds)} \n"
          f"MAE: {mean_absolute_error(Y, preds)} \n"
          f"MAPE: {mean_absolute_percentage_error(Y, preds)} \n"
          f"RMSE: {root_mean_squared_error(Y, preds)} \n")
    
eval_with_metrics(model, X_val, Y_val)